# Ray RLlib: SAC on Pendulum-v1

Project path: `projects/pendulum-sac`

**Soft Actor-Critic** for continuous torque control — companion step 3 in the blueprint ladder.

**Setup (once)** from the repository root:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r projects/pendulum-sac/requirements.txt
```

Select that kernel, then run the cell below. Script twin: `python projects/pendulum-sac/train_pendulum_sac.py`.

Pendulum is slower than CartPole: ~**5–10 minutes** for 15 iterations. Returns start very negative (~−1000); success is becoming *less* negative (toward ~−200 / −100).

In [ ]:
import math
import warnings
from typing import Any

warnings.filterwarnings(
    "ignore",
    message=r".*RLModule\(config=\[RLModuleConfig object\]\).*",
    category=DeprecationWarning,
)

from ray.rllib.algorithms.sac.sac import SACConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig


def episode_return_mean(result: dict[str, Any]) -> float | None:
    env_runners = result.get("env_runners") or {}
    value = env_runners.get("episode_return_mean")
    if value is None:
        return None
    value_f = float(value)
    if math.isnan(value_f):
        return None
    return value_f


config = (
    SACConfig()
    .environment("Pendulum-v1")
    .env_runners(num_env_runners=1)
    .training(
        replay_buffer_config={
            "type": "PrioritizedEpisodeReplayBuffer",
            "capacity": 100_000,
            "alpha": 0.6,
            "beta": 0.4,
        },
        num_steps_sampled_before_learning_starts=1_000,
        twin_q=True,
        gamma=0.99,
        actor_lr=3e-4,
        critic_lr=3e-4,
        train_batch_size_per_learner=256,
    )
    .reporting(min_sample_timesteps_per_iteration=1_000)
    .rl_module(model_config=DefaultModelConfig(fcnet_hiddens=[256, 256]))
    .evaluation(evaluation_num_env_runners=1)
    .debugging(log_level="ERROR")
)

algo = config.build_algo()
try:
    for i in range(1, 16):
        result = algo.train()
        ret = episode_return_mean(result)
        steps = result.get("num_env_steps_sampled_lifetime")
        if ret is not None:
            print(f"iter={i}  episode_return_mean={ret:.1f}  env_steps={steps}")
        else:
            print(
                f"iter={i}  episode_return_mean=n/a  env_steps={steps}  "
                "(no completed episodes in metric window)"
            )

    eval_result = algo.evaluate()
    eval_ret = episode_return_mean(eval_result)
    print(
        f"evaluate  episode_return_mean={eval_ret:.1f}"
        if eval_ret is not None
        else "evaluate  episode_return_mean=n/a"
    )
finally:
    algo.stop()

## What this teaches

| Idea | In this notebook |
| --- | --- |
| Continuous actions | Torque in \([-2, 2]\) on Pendulum |
| Off-policy actor-critic | SAC + twin Q + replay |
| Metric windows | `min_sample_timesteps_per_iteration` avoids empty/NaN episode stats |

Next: [Multi-Agent CartPole](../multiagent-cartpole/multiagent_cartpole.ipynb) · [Project README](README.md)